# 1. End-to-End Content Reranker Baseline Pipeline: Combined Candidates -> Content Similarity

This notebook runs the whole baseline pipeline by itself. It does not require any other notebook
to be executed first, and it does not read or write intermediate candidate files.

- **Candidate generation**: rebuilds the combined generator from shared project artifacts in
  memory. It builds 200 rolling prior-hour click-popularity candidates and 250 content
  text/entity cosine candidates per request.
- **Reranking**: reorders those 450 candidates by cosine similarity to the user's causal
  clicked-history profile. The run uses the fixed `content_fusion_text_0.50` rule; the other
  text, entity and fusion rules remain defined and can be compared by setting
  `SELECTED_TRIAL = None`.

The final Top-K list is evaluated with all metrics in `cg_evaluation.py` at each cutoff in
`K_VALUES = (5, 10, 30)`, plus MRR@k and nDCG@k from `rr_evaluation.py`. The saved final
Top-K list uses `K = max(K_VALUES) = 30`.

Saved artifacts are only the causal-feature replay cache, the final Top-K list and the metric
tables. Candidate lists and scores stay in memory.

**Labels are incomplete: evaluation counts known clicks only.** Only articles the logging
policy showed have a click label. A generated article that was never shown has *unknown*
relevance, so it is **not** a confirmed negative. In the metrics below it can never be
credited as a click, and no denominator treats it as a negative. It does, however, still
occupy a Top-K slot and can push a known-click article down or out. Every known click stays
in the denominator, so a click the generator or reranker left out of the Top-K is a miss.

The content rule is fixed to `content_fusion_text_0.50` by `SELECTED_TRIAL`. Set
`SELECTED_TRIAL = None` to compare all five rules on validation.

## 0. Configure

**This is the only cell you need to edit.** The defaults run the full experiment on the whole
dataset. After editing, run the whole notebook with **Runtime > Run all**.

In [ ]:
# ---- Where to run ----------------------------------------------------------------------------
# "colab" = Google Colab with Google Drive, "local" = this computer, "auto" = detect automatically.
RUN_ENV = "auto"

# ---- Paths -------------------------------------------------------------------------------------
# Project folder: the folder holding the code (this repository).
# On Colab, add a shortcut to the shared `submission` folder in My Drive first. The code is its
# `4.Code` folder; it is also found when the shortcut sits one folder deeper in My Drive.
COLAB_PROJECT_PATH = "/content/drive/MyDrive/submission/4.Code"
LOCAL_PROJECT_PATH = None  # None = the repository this notebook is opened from

# Data path: the prepared data, set separately for Colab and local runs. It must contain
# 1_Data_Preparation/processed/, 3_Representations/artifacts/ and
# 4_Feature_Engineering/artifacts/prequential-all/.
# Metrics are written to <data path>/results/.
# On Colab, the zipped data in <submission>/2.Feature_Engineering is unpacked into the data
# path on the first run (a few minutes); later runs reuse it.
COLAB_DATA_PATH = "/content/drive/MyDrive/submission/2.Feature_Engineering"
LOCAL_DATA_PATH = (
    "/Users/numkoos/Downloads/v3-mindsmall"  # used locally; None = the project folder
)

# ---- Experiment --------------------------------------------------------------------------------
# Split to evaluate: "test" for the final results, or "validation".
TARGET_SPLIT = "test"

# Target impressions to evaluate. None = every impression in the split (default).
# A number, e.g. 5_000, evaluates a random sample instead for a quicker run.
EVAL_IMPRESSIONS = None

## 1.1 Load Packages

In [33]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

## 1.2 Set Up Paths

The run mode, project folder, data path and model path all come from the **Configure** cell
(section 0). On Colab the notebook mounts Google Drive and finds the shared project folder at
`COLAB_PROJECT_PATH`, or one folder down in My Drive. The data folders are checked up front, so a
wrong `DATA_PATH` fails here with the missing folder named.

In [34]:
# Run mode and paths come from the Configure cell (section 0)
ENVIRONMENT = RUN_ENV.lower()
if ENVIRONMENT == "auto":
    ENVIRONMENT = "colab" if "google.colab" in sys.modules else "local"
if ENVIRONMENT not in {"colab", "local"}:
    raise ValueError('RUN_ENV must be "auto", "colab" or "local"')

if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    requested = Path(COLAB_PROJECT_PATH)
    my_drive = Path("/content/drive/MyDrive")
    CODE_ROOT = next(
        (
            p
            for p in [
                requested,
                *my_drive.glob(f"*/{requested.name}"),
                *my_drive.glob(f"*/*/{requested.name}"),
            ]
            if (p / "run_all.py").is_file()
        ),
        None,
    )
    if CODE_ROOT is None:
        raise FileNotFoundError(
            f"Could not find the project folder at {requested}. In Google Drive, right-click the shared "
            "`submission` folder > Organise > Add shortcut > My Drive (or fix COLAB_PROJECT_PATH), then rerun."
        )
elif LOCAL_PROJECT_PATH:
    CODE_ROOT = Path(LOCAL_PROJECT_PATH).expanduser().resolve()
else:
    CODE_ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
    )
ROOT = CODE_ROOT
DATA_PATH = COLAB_DATA_PATH if ENVIRONMENT == "colab" else LOCAL_DATA_PATH
DATA_ROOT = (
    Path(DATA_PATH or os.environ.get("NEWSREC_DATA_ROOT") or CODE_ROOT)
    .expanduser()
    .resolve()
)

REQUIRED_DATA = [
    "1_Data_Preparation/processed",
    "3_Representations/artifacts",
    "4_Feature_Engineering/artifacts/prequential-all",
]
# On Colab, unpack the zipped data from <submission>/2.Feature_Engineering on the first run.
# A marker file per archive means an interrupted unpack is redone rather than half-used.
SUBMISSION_ROOT = CODE_ROOT.parent if ENVIRONMENT == "colab" else None
BUNDLE_DIR = SUBMISSION_ROOT / "2.Feature_Engineering" if SUBMISSION_ROOT else None
if REQUIRED_DATA and BUNDLE_DIR is not None and BUNDLE_DIR.is_dir():
    import zipfile

    for archive in sorted(BUNDLE_DIR.glob("*.zip")):
        marker = DATA_ROOT / f".unpacked_{archive.stem}"
        if not marker.exists():
            print(f"Unpacking {archive.name} into {DATA_ROOT} (first run only) ...")
            with zipfile.ZipFile(archive) as bundle:
                bundle.extractall(DATA_ROOT)
            marker.touch()

missing_data = [
    str(DATA_ROOT / sub) for sub in REQUIRED_DATA if not (DATA_ROOT / sub).exists()
]
if missing_data:
    raise FileNotFoundError(
        f"{ENVIRONMENT.upper()}_DATA_PATH is missing these folders:\n"
        + "\n".join(missing_data)
    )
REPLAY_ROOT = DATA_ROOT

PROCESSED = REPLAY_ROOT / "1_Data_Preparation" / "processed"
REPRESENTATIONS = REPLAY_ROOT / "3_Representations" / "artifacts"
RUN = REPLAY_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
OUTPUT_DIR = REPLAY_ROOT / "5_Models" / "5_03_Pipeline_Evaluation" / "artifacts"
MODEL_DIR = OUTPUT_DIR / "baseline_content_popularity_combined_content_similarity"
RESULTS = REPLAY_ROOT / "results"
REPLAY_OUTPUT = OUTPUT_DIR / "replayed_features"

sys.path.insert(0, str(ROOT / "5_Models" / "5_01_Candidate_Generation"))
sys.path.insert(0, str(ROOT / "5_Models" / "5_02_Reranking"))
sys.path.insert(0, str(ROOT / "src"))

from cg_evaluation import CandidateEvaluator
from rr_evaluation import evaluate_retrieved_candidates

from newsrec.common import DEFAULT_SETTINGS
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.ranking import TemporalState, build_features, feature_schema
from newsrec.representations import Bundle

for path in (OUTPUT_DIR, MODEL_DIR, RESULTS, REPLAY_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

print("Repository:", ROOT)
print("Data path:", DATA_ROOT)
print("Model folder:", MODEL_DIR)

Repository: C:\Users\Yong Rui\BT4222-Group-6-
Replay root: C:\Users\Yong Rui\BT4222-Group-6-
Model folder: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\baseline_content_popularity_combined_content_similarity


## 1.3 Configuration

In [35]:
K_VALUES = (5, 10, 30)
K = max(K_VALUES)
SEED = 4222

CANDIDATE_MODEL_NAME = "baseline_content_popularity_combined"
POPULAR_TOP_N = 200
CONTENT_TOP_N = 250
CANDIDATE_CAP = POPULAR_TOP_N + CONTENT_TOP_N
HOUR_NS = 3_600 * 10**9

BASE_MODEL_NAME = f"{CANDIDATE_MODEL_NAME}_content_similarity"
SELECTION_PATH = (
    RESULTS
    / f"pipeline_{CANDIDATE_MODEL_NAME}_content_similarity_selection_validation.json"
)

CONTENT_COLUMNS = [
    "smoothed_prior_article_ctr",
    "text_cosine",
    "entity_cosine",
    "text_match_valid",
    "entity_match_valid",
    "no_text_profile",
    "no_entity_profile",
]
MISSING_SCORE = -2.0

TRIALS = {
    "content_text_cosine": {"kind": "text"},
    "content_entity_cosine": {"kind": "entity"},
    "content_fusion_text_0.25": {"kind": "fusion", "text_weight": 0.25},
    "content_fusion_text_0.50": {"kind": "fusion", "text_weight": 0.50},
    "content_fusion_text_0.75": {"kind": "fusion", "text_weight": 0.75},
}
SELECTED_TRIAL = "content_fusion_text_0.50"  # None = compare all trials on validation
SELECTION_METRIC = f"ndcg_at_{K}"
IMPRESSION_TAG = "all" if EVAL_IMPRESSIONS is None else f"{EVAL_IMPRESSIONS}imp"
CANDIDATE_CONFIG = {
    "popular_top_n": POPULAR_TOP_N,
    "content_top_n": CONTENT_TOP_N,
    "candidate_cap": CANDIDATE_CAP,
    "source_splits": (
        ["train", "validation"]
        if TARGET_SPLIT == "validation"
        else ["train", "validation", "test"]
    ),
    "seed": SEED,
    "target_split": TARGET_SPLIT,
    "eval_impressions": EVAL_IMPRESSIONS,
}

if TARGET_SPLIT not in {"validation", "test"}:
    raise ValueError("TARGET_SPLIT must be validation or test")
if SELECTED_TRIAL is not None and SELECTED_TRIAL not in TRIALS:
    raise ValueError(f"SELECTED_TRIAL must be one of {sorted(TRIALS)}")
if K > CANDIDATE_CAP:
    raise ValueError("K cannot exceed the candidate cap")

schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
if feature_schema(DEFAULT_SETTINGS)["model_columns"] != FEATURES:
    raise ValueError(
        "The feature code and the saved feature run disagree on the schema"
    )
missing = set(CONTENT_COLUMNS) - set(FEATURES)
if missing:
    raise ValueError(
        f"Content columns missing from the feature schema: {sorted(missing)}"
    )

print("Target split:", TARGET_SPLIT)
print("Reported K values:", K_VALUES)
print("Final K:", K)
print("Selected content rule:", SELECTED_TRIAL)
print("Evaluation impressions:", EVAL_IMPRESSIONS)
print(
    f"Candidate list: {POPULAR_TOP_N} popular + {CONTENT_TOP_N} content = {CANDIDATE_CAP}"
)

Target split: test
Reported K values: (5, 10, 30)
Final K: 30
Selected content rule: content_fusion_text_0.50
Evaluation impressions: None
Candidate list: 200 popular + 250 content = 450


## 1.4 Build the Candidate Generator Inputs

The baseline generator is deterministic: it rebuilds the rolling popularity order and the
content cosine channels from the shared stage 1, 3 and 4 artifacts. Request histories and
timestamps come from `request_profiles_<split>`. Only the sampled target requests are built,
so their candidates stay in memory.

In [36]:
index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code"],
).sort("article_row")
if not np.array_equal(index["article_row"].to_numpy(), np.arange(1, index.height + 1)):
    raise ValueError("article_index rows must be 1..N with no gaps")

N_ITEMS = index.height + 1
ROWS = index["article_row"].to_numpy().astype(np.int64)
CODES = index["news_code"].to_numpy().astype(np.int64)
ROW_OF_CODE = np.zeros(int(CODES.max()) + 1, dtype=np.int64)
ROW_OF_CODE[CODES] = ROWS
ROW_TO_CODE = np.zeros(N_ITEMS, dtype=np.int32)
ROW_TO_CODE[ROWS] = CODES.astype(np.int32)

availability = pl.read_parquet(
    REPRESENTATIONS / "article_availability.parquet",
    columns=["news_code", "available_from"],
)
avail_meta = index.join(availability, on="news_code", how="left")
avail_rows_meta = avail_meta["article_row"].to_numpy().astype(np.int64)
avail_values = (
    avail_meta["available_from"]
    .dt.epoch("ns")
    .fill_null(np.iinfo(np.int64).max)
    .to_numpy()
    .astype(np.int64)
)
AVAIL_NS = np.full(N_ITEMS, np.iinfo(np.int64).max, dtype=np.int64)
AVAIL_NS[avail_rows_meta] = avail_values

first_ts = (
    pl.scan_parquet(PROCESSED / "candidates_train.parquet")
    .select(pl.col("timestamp").min())
    .collect()
    .item()
)
T0_NS = int(
    np.datetime64(first_ts, "ns")
    .astype("datetime64[D]")
    .astype("datetime64[ns]")
    .astype(np.int64)
)

impressions = pl.read_parquet(
    PROCESSED / f"impressions_{TARGET_SPLIT}.parquet",
    columns=["impression_uid", "timestamp"],
)
profile_dir = RUN / f"request_profiles_{TARGET_SPLIT}"
profile_parts = sorted(profile_dir.glob("*.parquet"))
if not profile_parts:
    raise FileNotFoundError(f"No request profiles under {profile_dir}")

history_col = (
    "full_history_rows"
    if "full_history_rows" in pl.read_parquet_schema(profile_parts[0])
    else "history_rows"
)
profiles = pl.read_parquet(
    profile_parts,
    columns=["impression_uid", "timestamp", history_col],
).sort(["timestamp", "impression_uid"])

missing = (
    impressions.select("impression_uid")
    .unique()
    .join(
        profiles.select("impression_uid"),
        on="impression_uid",
        how="anti",
    )
)
if missing.height:
    raise ValueError(f"{missing.height:,} impressions have no request profile")

ALL_UID = profiles["impression_uid"].to_numpy().astype(np.int64)
ALL_TIMESTAMPS = profiles["timestamp"].to_numpy()
ALL_TS = profiles["timestamp"].dt.epoch("ns").to_numpy().astype(np.int64)
ALL_HOUR = (ALL_TS - T0_NS) // HOUR_NS
ALL_HISTORY = [
    [int(row) for row in history] if history else []
    for history in profiles[history_col].to_list()
]

if EVAL_IMPRESSIONS is None:
    target_indices = np.arange(ALL_UID.size)
else:
    all_uids = np.sort(np.unique(ALL_UID))
    chosen = np.random.default_rng(SEED).choice(
        all_uids, size=min(EVAL_IMPRESSIONS, all_uids.size), replace=False
    )
    target_indices = np.flatnonzero(np.isin(ALL_UID, chosen))

uid = ALL_UID[target_indices]
timestamps = ALL_TIMESTAMPS[target_indices]
ts = ALL_TS[target_indices]
hour = ALL_HOUR[target_indices]
history_rows = [ALL_HISTORY[i] for i in target_indices]

if uid.size == 0:
    raise ValueError("No target requests were selected")
if np.any(hour < 0):
    raise ValueError("Target impressions fall outside the hourly state range")

print(f"Catalogue articles: {N_ITEMS - 1:,}")
print(f"Target requests: {uid.size:,} of {ALL_UID.size:,}")
print(f"Reference hour: {T0_NS}")

Catalogue articles: 51,282
Target requests: 30,270 of 30,270
Reference hour: 1573257600000000000


## 1.5 Build the Rolling Popularity Channel

The hourly state aggregates logged candidate slates by `(news_code, hour)`. A request in hour
`h` reads only row `h - 1`, so popularity is strictly causal. The channel removes unavailable
articles and every article in the request history, then keeps the first 200.

In [37]:
SOURCE_SPLITS = CANDIDATE_CONFIG["source_splits"]

hourly = (
    pl.concat(
        [
            pl.scan_parquet(PROCESSED / f"candidates_{split}.parquet").select(
                "news_code",
                "clicked",
                ((pl.col("timestamp").dt.epoch("ns") - T0_NS) // HOUR_NS).alias("hour"),
            )
            for split in SOURCE_SPLITS
        ]
    )
    .group_by("news_code", "hour")
    .agg(
        exp=pl.len(),
        clk=pl.col("clicked").cast(pl.Int64).sum(),
    )
    .collect()
)

source_rows = ROW_OF_CODE[hourly["news_code"].to_numpy().astype(np.int64)]
hour_idx = hourly["hour"].to_numpy()
N_HOURS = int(hour_idx.max()) + 1
if hour.min() < 0 or hour.max() >= N_HOURS:
    raise ValueError("Target impressions fall outside the hourly state range")

EXPOSURES = np.zeros((N_HOURS, N_ITEMS), dtype=np.int32)
CLICKS = np.zeros((N_HOURS, N_ITEMS), dtype=np.int32)
EXPOSURES[hour_idx, source_rows] = hourly["exp"].to_numpy()
CLICKS[hour_idx, source_rows] = hourly["clk"].to_numpy()


def score_snapshot(hour_):
    if hour_ == 0:
        zero = np.zeros(N_ITEMS, dtype=np.int32)
        return zero, zero
    return CLICKS[hour_ - 1], EXPOSURES[hour_ - 1]


def eligible_order(hour_):
    hour_end = T0_NS + (hour_ + 1) * HOUR_NS
    eligible = np.flatnonzero(AVAIL_NS <= hour_end)
    score, tiebreak = score_snapshot(hour_)
    return eligible[
        np.lexsort((ROW_TO_CODE[eligible], -tiebreak[eligible], -score[eligible]))
    ]


max_history = max((len(rows_) for rows_ in history_rows), default=0)
popular_pool_depth = POPULAR_TOP_N + max_history + 4_000
popular_rows = np.zeros((uid.size, POPULAR_TOP_N), dtype=np.int64)

starts = np.flatnonzero(np.r_[True, hour[1:] != hour[:-1]])
ends = np.r_[starts[1:], len(hour)]

for hour_value, start, end in zip(
    hour[starts].tolist(), starts.tolist(), ends.tolist()
):
    idx = np.arange(start, end, dtype=np.int64)
    ranked = eligible_order(hour_value)
    pool = ranked[: min(ranked.size, popular_pool_depth)]

    lengths = np.array([len(history_rows[i]) for i in idx], dtype=np.int64)
    if lengths.sum():
        read_rows = np.concatenate(
            [
                np.asarray(history_rows[i], dtype=np.int64)
                for i in idx
                if history_rows[i]
            ]
        )
        history_keys = (
            np.repeat(np.arange(idx.size, dtype=np.int64), lengths) * N_ITEMS
            + read_rows
        )
        in_history = np.isin(
            np.arange(idx.size, dtype=np.int64)[:, None] * N_ITEMS + pool[None, :],
            history_keys,
        )
    else:
        in_history = np.zeros((idx.size, pool.size), dtype=bool)

    available = AVAIL_NS[pool][None, :] <= ts[idx][:, None]
    keep = available & ~in_history
    take = keep & (np.cumsum(keep, axis=1) <= POPULAR_TOP_N)
    full = take.sum(axis=1) == POPULAR_TOP_N
    popular_rows[idx[full]] = pool[np.nonzero(take[full])[1]].reshape(-1, POPULAR_TOP_N)

    # Rare fallback: scan the full ranking when the fixed pool was too shallow.
    for local in np.flatnonzero(~full):
        request = idx[local]
        history_set = set(history_rows[request])
        chosen = []
        for article_row in ranked:
            if (
                AVAIL_NS[article_row] <= ts[request]
                and int(article_row) not in history_set
            ):
                chosen.append(int(article_row))
                if len(chosen) == POPULAR_TOP_N:
                    break
        if len(chosen) != POPULAR_TOP_N:
            raise ValueError(
                f"Request {uid[request]} has fewer than {POPULAR_TOP_N} popular articles"
            )
        popular_rows[request] = chosen

if (popular_rows == 0).any():
    raise ValueError("The popular channel did not fill every request")

popular_codes = ROW_TO_CODE[popular_rows]
print(f"Popular candidates: {popular_rows.shape[0]:,} requests x {POPULAR_TOP_N}")

Popular candidates: 30,270 requests x 200


## 1.6 Build the Content Channel and Candidate Lists

Content candidates are the next 250 articles by text/entity cosine to the mean-pooled history
profile, excluding history and the popularity channel. When a profile has no valid vector,
the channel falls back to the global training-popularity order. The final in-memory table is
the standard long format: `cutoff = 1..200` popularity and `201..450` content.

In [38]:
text = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
entity = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")

counts = pl.read_parquet(
    PROCESSED / "news_catalogue.parquet",
    columns=["news_code", "exposures_train"],
)
popularity = counts.select(
    "news_code", pl.col("exposures_train").alias("popularity_score")
)

catalogue = index.join(availability, on="news_code", how="left").join(
    popularity, on="news_code", how="left"
)
ordered = catalogue.sort(["available_from", "news_code"])

article_rows_sorted = ordered["article_row"].to_numpy().astype(np.int64)
news_sorted = ordered["news_code"].to_numpy().astype(np.int64)
available_sorted = ordered["available_from"].to_numpy().astype("datetime64[ns]")
popularity_sorted = ordered["popularity_score"].fill_null(0).to_numpy().astype(np.int64)
text_sorted = np.asarray(text[article_rows_sorted], dtype=np.float32)
entity_sorted = np.asarray(entity[article_rows_sorted], dtype=np.float32)
text_article_valid = np.linalg.norm(text_sorted, axis=1) > 0
entity_article_valid = np.linalg.norm(entity_sorted, axis=1) > 0

pos_in_sorted = np.empty(N_ITEMS, dtype=np.int64)
pos_in_sorted[article_rows_sorted] = np.arange(len(article_rows_sorted), dtype=np.int64)
global_popularity_order = np.lexsort((news_sorted, -popularity_sorted))


def pool_profiles(histories, matrix, block=10_000):
    """Mean-pool valid article vectors per request and L2-normalise."""
    matrix = np.asarray(matrix)
    n = len(histories)
    dim = matrix.shape[1]
    result = np.zeros((n, dim), dtype=np.float32)

    for start in range(0, n, block):
        chunk = histories[start : start + block]
        lengths = np.fromiter(
            (len(rows_) for rows_ in chunk), dtype=np.int64, count=len(chunk)
        )
        parts = [np.asarray(rows_, dtype=np.int64) for rows_ in chunk if len(rows_)]
        if not parts:
            continue
        flat = np.concatenate(parts)
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12

        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        group_valid = groups[valid]
        vector_valid = vectors[valid]
        present_counts = np.bincount(group_valid, minlength=len(chunk))
        sums = np.empty((len(chunk), dim), dtype=np.float32)
        for column in range(dim):
            sums[:, column] = np.bincount(
                group_valid,
                weights=vector_valid[:, column],
                minlength=len(chunk),
            )
        present = present_counts > 0
        pooled = sums[present] / present_counts[present, None]
        norms = np.linalg.norm(pooled, axis=1)
        pooled /= np.where(norms > 0, norms, 1.0)[:, None]
        result[start : start + len(chunk)][present] = pooled

    return result


text_profiles = pool_profiles(history_rows, text)
entity_profiles = pool_profiles(history_rows, entity)
text_profile_valid = np.linalg.norm(text_profiles, axis=1) > 0
entity_profile_valid = np.linalg.norm(entity_profiles, axis=1) > 0


def fuse_scores(text_scores, entity_scores, text_available, entity_available):
    """Equal-weight fusion with valid-side renormalisation."""
    fused = np.full(text_scores.shape, -np.inf, dtype=np.float32)
    both = text_available & entity_available
    only_text = text_available & ~entity_available
    only_entity = entity_available & ~text_available
    fused[both] = 0.5 * (text_scores[both] + entity_scores[both])
    fused[only_text] = text_scores[only_text]
    fused[only_entity] = entity_scores[only_entity]
    return fused


def popularity_select(order, n_eligible, excluded, k):
    """Take popular eligible positions from a precomputed global order."""
    excluded_set = set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position < n_eligible and position not in excluded_set:
            selected.append(position)
            if len(selected) == k:
                break
    return np.asarray(selected, dtype=np.int64)


def select_top_k(scores, popularity_, news_code, k, excluded, fallback_order=None):
    """Deterministic Top-k by score, popularity, then news_code."""
    n = scores.size
    if fallback_order is not None:
        return popularity_select(fallback_order, n, excluded, k)

    positions = np.asarray(excluded, dtype=np.int64)
    if positions.size:
        scores = scores.copy()
        scores[positions] = -np.inf

    frontier = np.argpartition(-scores, k - 1)[:k]
    threshold = scores[frontier].min()
    above = np.flatnonzero(scores > threshold)
    need = k - above.size
    ties = np.flatnonzero(scores == threshold)
    if ties.size > need:
        tie_order = np.lexsort((news_code[ties], -popularity_[ties]))
        ties = ties[tie_order[:need]]
    chosen_positions = np.r_[above, ties]

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity_[chosen_positions],
            -scores[chosen_positions],
        )
    )
    return chosen_positions[order].astype(np.int64)


popular_positions = pos_in_sorted[popular_rows]
content_rows = np.zeros((uid.size, CONTENT_TOP_N), dtype=np.int64)

starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
batches = list(zip(starts.tolist(), ends.tolist()))

print(f"Scoring requests: {uid.size:,}")

for batch_no, (start, end) in enumerate(batches, 1):
    timestamp = timestamps[start]
    n_eligible = int(np.searchsorted(available_sorted, timestamp, side="right"))

    popularity_batch = popularity_sorted[:n_eligible]
    news_batch = news_sorted[:n_eligible]
    text_batch = text_sorted[:n_eligible]
    entity_batch = entity_sorted[:n_eligible]
    text_article_available = text_article_valid[:n_eligible]
    entity_article_available = entity_article_valid[:n_eligible]

    text_scores = text_profiles[start:end] @ text_batch.T
    entity_scores = entity_profiles[start:end] @ entity_batch.T

    text_available = (
        text_profile_valid[start:end, None] & text_article_available[None, :]
    )
    entity_available = (
        entity_profile_valid[start:end, None] & entity_article_available[None, :]
    )
    text_scores[~text_available] = -np.inf
    entity_scores[~entity_available] = -np.inf
    fused_scores = fuse_scores(
        text_scores, entity_scores, text_available, entity_available
    )

    for local, profile_row in enumerate(range(start, end)):
        excluded_parts = []

        history = history_rows[profile_row]
        if history:
            history_pos = pos_in_sorted[np.asarray(history, dtype=np.int64)]
            excluded_parts.append(history_pos[history_pos < n_eligible])

        popular_pos = popular_positions[profile_row]
        excluded_parts.append(popular_pos[popular_pos < n_eligible])
        excluded = (
            np.concatenate(excluded_parts)
            if excluded_parts
            else np.empty(0, dtype=np.int64)
        )

        fallback = (
            None
            if (text_profile_valid[profile_row] or entity_profile_valid[profile_row])
            else global_popularity_order
        )
        content_rows[profile_row] = select_top_k(
            fused_scores[local],
            popularity_batch,
            news_batch,
            CONTENT_TOP_N,
            excluded,
            fallback,
        )

    if batch_no % 5000 == 0 or batch_no == len(batches):
        print(f"Scored {batch_no:,} / {len(batches):,} timestamp batches", flush=True)

content_codes = news_sorted[content_rows]
combined_codes = np.concatenate([popular_codes, content_codes], axis=1)

if combined_codes.shape != (uid.size, CANDIDATE_CAP):
    raise ValueError("Combined candidate matrix has the wrong shape")
ordered_codes = np.sort(combined_codes, axis=1)
if (ordered_codes[:, 1:] == ordered_codes[:, :-1]).any():
    raise ValueError("Combined lists contain duplicate articles")

cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
candidates = pl.DataFrame(
    {
        "impression_uid": np.repeat(uid, CANDIDATE_CAP).astype(np.int64),
        "cutoff": np.tile(cutoff_values, uid.size),
        "news_code": combined_codes.reshape(-1).astype(np.int64),
    }
).sort(["impression_uid", "cutoff"])

selected_uids = candidates["impression_uid"].unique().sort()
n_selected = selected_uids.len()

print(f"Popular candidates: {popular_codes.shape[1]}")
print(f"Content candidates: {content_codes.shape[1]}")
print(f"Target candidates: {n_selected:,} impressions / {candidates.height:,} rows")

Scoring requests: 30,270


KeyboardInterrupt: 

# 2. Replay Causal Features

The generated candidates are not logged slate entries, so most of them have no row in
`ranking_features_*`. The causal timeline is replayed to build the same content features used
by `RR_content_similarity_baseline.ipynb`. Features for a request use only history strictly
before its timestamp, and candidates never update the state.

The cache holds only the seven columns the content rule needs, so re-runs reuse it.

In [ ]:
def replay_content_features(split, candidate_frame):
    """Replay the causal timeline and cache the content features for generated candidates."""
    impression_count = candidate_frame["impression_uid"].n_unique()
    cache_path = (
        REPLAY_OUTPUT / f"{split}_{CANDIDATE_MODEL_NAME}_{impression_count}imp_"
        f"{candidate_frame.height}rows_content_features.parquet"
    )
    if cache_path.exists():
        print("Reusing replayed features:", cache_path.name)
        return cache_path

    frame = (
        candidate_frame.sort(["impression_uid", "cutoff"])
        .to_pandas()
        .reset_index(drop=True)
    )
    rows_by_impression = frame.groupby("impression_uid", sort=False).indices
    target_ids = set(rows_by_impression)
    replay_splits = tuple(
        ("train", "validation", "test")[
            : ("train", "validation", "test").index(split) + 1
        ]
    )

    bundle = Bundle(REPLAY_ROOT, DEFAULT_SETTINGS)
    state = TemporalState(bundle, *time_horizon(REPLAY_ROOT), DEFAULT_SETTINGS)
    partial_path = cache_path.with_suffix(".partial.parquet")
    writer, pending, impressions, started = None, [], 0, time.monotonic()
    output_columns = ["impression_uid", "news_code", "cutoff", *CONTENT_COLUMNS]

    def flush():
        nonlocal writer, pending
        if not pending:
            return
        out = pd.concat(pending, ignore_index=True)[output_columns]
        table = pa.Table.from_pandas(out, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(partial_path, table.schema, compression="zstd")
        writer.write_table(table)
        pending = []

    print(
        f"Replaying {split}: {len(frame):,} candidates for "
        f"{len(rows_by_impression):,} impressions"
    )
    try:
        for timestamp, requests, events in replay_batches(
            REPLAY_ROOT, DEFAULT_SETTINGS, splits=replay_splits
        ):
            state.begin(requests, timestamp)
            selected = requests[
                requests["split"].eq(split)
                & requests["impression_uid"].isin(target_ids)
            ]
            if len(selected):
                pairs = frame.iloc[
                    np.concatenate(
                        [rows_by_impression[uid] for uid in selected["impression_uid"]]
                    )
                ]
                features = build_features(
                    selected, pairs, state, bundle, DEFAULT_SETTINGS, "retrieved"
                ).drop(columns=["cutoff"], errors="ignore")
                pending.append(
                    features.merge(
                        pairs[["impression_uid", "news_code", "cutoff"]],
                        on=["impression_uid", "news_code"],
                        validate="one_to_one",
                    )
                )
                impressions += len(selected)
                if sum(len(chunk) for chunk in pending) >= 50_000:
                    flush()
                    print(
                        f"  {impressions:,}/{len(rows_by_impression):,} impressions "
                        f"({(time.monotonic() - started) / 60:.1f} min)",
                        flush=True,
                    )
            state.finish(requests, events, timestamp, set(replay_splits))
        flush()
    finally:
        if writer is not None:
            writer.close()
        state.close()

    if impressions != len(rows_by_impression):
        raise ValueError(
            f"Replayed {impressions:,} impressions but {len(rows_by_impression):,} were selected"
        )
    partial_path.replace(cache_path)
    print(f"Saved replayed features: {cache_path.name}")
    return cache_path


features_path = replay_content_features(TARGET_SPLIT, candidates)

Reusing replayed features: test_baseline_content_popularity_combined_30270imp_13621500rows_content_features.parquet


# 3. Content Similarity Reranking

The scoring rule is copied unchanged from `RR_content_similarity_baseline.ipynb`. A candidate
with no valid vector scores `-2`, below every valid cosine in `[-1, 1]`. If the user has no
profile at all, the rule falls back to the causal smoothed article CTR.

The selected rule reorders all 450 candidates by score descending, with `news_code`
ascending as the deterministic tie-break, and keeps ranks `1..K` where `K = max(K_VALUES)`.
Metrics are then reported at each cutoff in `K_VALUES`.

In [ ]:
def content_score(frame, trial):
    """Score candidates with one text, entity or fusion rule."""
    prior = frame["smoothed_prior_article_ctr"].to_numpy().astype(float)
    text = frame["text_cosine"].to_numpy().astype(float)
    entity = frame["entity_cosine"].to_numpy().astype(float)
    text_valid = frame["text_match_valid"].to_numpy().astype(bool)
    entity_valid = frame["entity_match_valid"].to_numpy().astype(bool)
    no_text_profile = frame["no_text_profile"].to_numpy().astype(bool)
    no_entity_profile = frame["no_entity_profile"].to_numpy().astype(bool)

    if trial["kind"] == "text":
        return np.where(
            no_text_profile, prior, np.where(text_valid, text, MISSING_SCORE)
        )
    if trial["kind"] == "entity":
        return np.where(
            no_entity_profile, prior, np.where(entity_valid, entity, MISSING_SCORE)
        )

    alpha = trial["text_weight"]
    blended = np.where(
        text_valid & entity_valid,
        alpha * text + (1 - alpha) * entity,
        np.where(text_valid, text, np.where(entity_valid, entity, MISSING_SCORE)),
    )
    return np.where(no_text_profile & no_entity_profile, prior, blended)


def rank_top_k(scored, trial):
    """Apply one content rule and keep each impression's final Top-K list."""
    scores = content_score(scored, trial)
    return (
        scored.with_columns(pl.Series("score", scores, dtype=pl.Float64))
        .sort(["impression_uid", "score", "news_code"], descending=[False, True, False])
        .with_columns(
            pl.int_range(1, pl.len() + 1)
            .over("impression_uid")
            .cast(pl.Int16)
            .alias("cutoff")
        )
        .filter(pl.col("cutoff") <= K)
        .select("impression_uid", "cutoff", "news_code", "score")
    )


replayed = pl.read_parquet(
    features_path, columns=["impression_uid", "news_code", *CONTENT_COLUMNS]
)
scored_candidates = candidates.join(
    replayed, on=["impression_uid", "news_code"], how="left"
)
if scored_candidates.select(
    pl.any_horizontal(pl.col(CONTENT_COLUMNS).is_null()).sum()
).item():
    raise ValueError("Some generated candidates are missing replayed content features")

# 4. Evaluate the Pipeline

Three groups are reported for the reranked list at each cutoff in `K_VALUES`:

- **`cg_evaluation.py` metrics at each `M = k`**: pooled and impression-mean recall, hit rate,
  long-tail recall, cold-article recall, novelty, catalogue coverage and personalisation.
- **MRR@k**: reciprocal rank of the first known click in the list truncated at `k`.
- **nDCG@k**: binary-click nDCG over the list truncated at `k`.

How the denominators work:

- `recall_pooled` divides by every known click in the evaluated impressions.
- `recall_impression_mean` averages hits/clicks only over impressions with a click.
- `hit_rate` divides by **all** evaluated impressions, including those with no click.
- `recall_at_k`, `mrr` and `nDCG@k` from `rr_evaluation.py` use only impressions with at
  least one known click.
- nDCG's ideal gain uses **every** known click in the impression, so clicks outside the
  cutoff lower the score even though they were never retrieved.

A retrieved article with no label is treated as "not a known click"
(`evaluate_retrieved_candidates` fills `clicked` with 0), but no metric divides by
unlabelled candidates, so it is not scored as a confirmed negative. It still occupies a
Top-K slot.

The content rule is fixed to `content_fusion_text_0.50` by `SELECTED_TRIAL`. Set
`SELECTED_TRIAL = None` to choose by validation nDCG@K.

In [ ]:
evaluator = CandidateEvaluator(root=REPLAY_ROOT, split=TARGET_SPLIT)

CG_METRICS = [
    "recall_pooled",
    "recall_impression_mean",
    "hit_rate",
    "long_tail_recall",
    "cold_article_recall",
    "novelty_bits",
    "catalogue_coverage",
    "personalisation",
]


def evaluate_pipeline(top_k, model_name, k):
    """Combine the cg_evaluation metrics at k with MRR@k and nDCG@k."""
    top_k = top_k.filter(pl.col("cutoff") <= k)
    cg = evaluator.evaluate(
        top_k.select("impression_uid", "cutoff", "news_code"),
        model_name=model_name,
        m_values=(k,),
        impression_uids=selected_uids,
    )
    cg_wide = (
        cg.pivot(on="metric", index=["model", "split", "m"], values="value")
        .rename({"m": "k"})
        .with_columns(pl.lit(n_selected).alias("impressions"))
    )
    rr = evaluate_retrieved_candidates(
        top_k.select("impression_uid", "news_code", "score"),
        feature_run=RUN,
        split=TARGET_SPLIT,
        score_column="score",
        model_name=model_name,
        k_values=(k,),
    ).select(
        "model",
        pl.col("impressions").alias("clicked_impressions"),
        "mrr",
        pl.col(f"ndcg_at_{k}").alias("ndcg"),
    )
    return cg_wide.join(rr, on="model", how="left")


if SELECTED_TRIAL is None:
    if TARGET_SPLIT == "validation":
        trial_names = list(TRIALS)
    else:
        if not SELECTION_PATH.exists():
            raise FileNotFoundError(
                f"Validation selection not found: {SELECTION_PATH}. Run this notebook "
                "with TARGET_SPLIT='validation' first, or set SELECTED_TRIAL."
            )
        trial_names = [json.loads(SELECTION_PATH.read_text())["selected_trial"]]
else:
    trial_names = [SELECTED_TRIAL]
    print("Using fixed content rule:", SELECTED_TRIAL)

top_k_by_trial = {}
rows = []
for name in trial_names:
    top_k_by_trial[name] = rank_top_k(scored_candidates, TRIALS[name])
    for k in K_VALUES:
        rows.append(evaluate_pipeline(top_k_by_trial[name], name, k))
    print("Evaluated:", name)

results = pl.concat(rows, how="diagonal_relaxed")

if TARGET_SPLIT == "validation" and SELECTED_TRIAL is None:
    selection_rows = results.filter(pl.col("k") == K)
    trial_results = selection_rows.filter(pl.col("model").is_in(trial_names)).sort(
        "ndcg", descending=True
    )
    selected_trial = trial_results["model"][0]
    SELECTION_PATH.write_text(
        json.dumps(
            {
                "split": "validation",
                "k": K,
                "k_values": list(K_VALUES),
                "selection_metric": SELECTION_METRIC,
                "selected_trial": selected_trial,
                "trial_scores": {
                    row["model"]: row["ndcg"]
                    for row in trial_results.iter_rows(named=True)
                },
                "eval_impressions": int(n_selected),
                "seed": SEED,
                "candidate_generator": CANDIDATE_MODEL_NAME,
                "candidate_config": CANDIDATE_CONFIG,
            },
            indent=2,
        )
    )
    print("Validation-selected content rule:", selected_trial)
elif TARGET_SPLIT == "test" and SELECTED_TRIAL is None:
    selected_trial = trial_names[0]
    print("Using validation-selected content rule:", selected_trial)
else:
    selected_trial = trial_names[0]
    print("Using fixed content rule:", selected_trial)

final_top_k = top_k_by_trial[selected_trial]
final_top_k_path = (
    OUTPUT_DIR / f"{BASE_MODEL_NAME}_{TARGET_SPLIT}_top{K}_{IMPRESSION_TAG}.parquet"
)
final_top_k.write_parquet(final_top_k_path)
print("Final Top-K saved to:", final_top_k_path)

config_path = MODEL_DIR / "config.json"
config_path.write_text(
    json.dumps(
        {
            "candidate_generator": CANDIDATE_MODEL_NAME,
            "candidate_config": CANDIDATE_CONFIG,
            "target_split": TARGET_SPLIT,
            "k_values": list(K_VALUES),
            "final_k": K,
            "selected_trial": selected_trial,
            "selection_mode": "fixed" if SELECTED_TRIAL is not None else "validation",
            "trials": TRIALS,
            "selection_metric": SELECTION_METRIC,
            "eval_impressions": EVAL_IMPRESSIONS,
            "seed": SEED,
            "metrics": {
                f"{row['model']}@{row['k']}": {
                    "ndcg": row["ndcg"],
                    "mrr": row["mrr"],
                }
                for row in results.iter_rows(named=True)
            },
        },
        indent=2,
    )
)
print("Config saved to:", config_path)

ordered = results.select(
    "model",
    "split",
    "k",
    "impressions",
    "clicked_impressions",
    "ndcg",
    "mrr",
    *CG_METRICS,
).sort(["model", "k"])
results_path = (
    RESULTS / f"pipeline_{CANDIDATE_MODEL_NAME}_content_similarity_"
    f"{TARGET_SPLIT}_{IMPRESSION_TAG}.csv"
)
ordered.write_csv(results_path)
print("Metrics saved to:", results_path)
display(ordered)

Using fixed content rule: content_fusion_text_0.50
Evaluated: content_fusion_text_0.50
Using fixed content rule: content_fusion_text_0.50
Final Top-K saved to: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\baseline_content_popularity_combined_content_similarity_test_top30_all.parquet
Config saved to: C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_03_Pipeline_Evaluation\artifacts\baseline_content_popularity_combined_content_similarity\config.json
Metrics saved to: C:\Users\Yong Rui\BT4222-Group-6-\results\pipeline_baseline_content_popularity_combined_content_similarity_test_all.csv


model,split,k,impressions,clicked_impressions,ndcg,mrr,recall_pooled,recall_impression_mean,hit_rate,long_tail_recall,cold_article_recall,novelty_bits,catalogue_coverage,personalisation
str,str,i64,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""content_fusion_text_0.50""","""test""",5,30270,30270,0.000491,0.000424,0.000705,0.000851,0.00109,0.000514,0.005495,15.704159,0.206622,0.987875
"""content_fusion_text_0.50""","""test""",10,30270,30270,0.000809,0.000613,0.001644,0.001743,0.002544,0.000926,0.005495,15.701255,0.295152,0.984834
"""content_fusion_text_0.50""","""test""",30,30270,30270,0.001549,0.000855,0.004399,0.004607,0.006772,0.002057,0.005495,15.699737,0.471959,0.977142


# 5. Notes and Limitations

- The notebook is self-contained: it rebuilds the popularity order and the content scores from
  stage 1, 3 and 4 artifacts. It does not read saved candidate files from another notebook.
- The popularity half is non-personalised: every request in the same hour gets the same base
  order, apart from availability and history filtering. The content half is already
  personalised, and the content reranker reorders the combined 450 candidates.
- All metrics are known-click metrics. A generated article that was never shown has unknown
  relevance: it cannot be credited as a click and is not a confirmed negative, but it still
  occupies a Top-K slot. The final list is therefore partly rewarded for matching what the
  logging policy showed, and an unshown article that would have been clicked is an unknown
  miss rather than a measured one.
- Metrics are reported at `K_VALUES = (5, 10, 30)`; the saved final list uses
  `K = max(K_VALUES) = 30`. `cg_evaluation.py` metrics at each `M = k` describe the list
  truncated at `k`, not the 450-candidate pool. `catalogue_coverage` and `personalisation`
  therefore measure the final list, not retrieval depth.
- The content rule defaults to `content_fusion_text_0.50` (`SELECTED_TRIAL`). Set
  `SELECTED_TRIAL = None` to compare all five rules on validation and save a selection
  for test.
- The test run uses all three source splits (`train`, `validation`, `test`) for the hourly
  popularity state, and reads `request_profiles_test`. It does not need a separate candidate
  generation notebook.